# Лабораторна #3: Word embeddings for text classification

У цій роботі ми використаємо word embeddings для тренування моделі класифікації текстів.

Маємо побачити, як word embeddings особливо сильно допомогають, коли тренувальних даних небагато (а їх майже завжди небагато).

In [39]:
import warnings
warnings.filterwarnings("ignore")

In [40]:
import lpnlp
lab = lpnlp.start(
    email="sviatoslav.humeniuk.shi.2023@lpnu.ua",
    lab="using_word_embeddings",
)

Удачі!


# GloVe

Повний GloVe містить 4,000,000 векторів і займає багато пам'яті. Щоб уникнути проблем з пам'ятю, залишимо лише 50,000 векторів найчастотніших слів. Це трохи знизить якість моделей, але це зараз не головне.

In [41]:
from gensim.models import KeyedVectors
glove = KeyedVectors.load("https://static.lp-nlp.com/files/glove-50k.bin")

# Bag-of-embeddings

Вектори слів, натреновані на великому корпусі текстів, використовують для представлення слів замість розріджених one-hot, які ми бачили в першій лабораторній. Word embeddings чудово працюють з нейронним мережами різноманітних архітектур. Але зараз ми розглянемо напростіше використання: логістичну регресію (так, знову) та мішок векторів (bag-of-embeddings).

В bag-of-embeddings ми усереднюємо вектори всіх слів, які входять в речення. Результат — вектор такої ж розмірності, як і вектор слова. Цей вектор кодує зміст усього речення. Звичайно, таке представлення не враховує порядок слів, рівноцінно ставиться до важливих та допоміжних слів, а тому "кодує" воно зміст речення вельми приблизно. Проте цього достатньо для багатьох простих задач.

## Токенізація

Для початку нам треба токенізувати корпус. Важливий момент: GloVe та інші word embeddings тренувалися кожен зі своїм токенізатором. Нам слід використовувати максимально схожий токенізатор. Інакше ми не зможемо знайти вектори для багатьох слів.

Насамперед, GloVe тренувалися на тексті, приведеному до нижнього регістру. Також розбіжності можуть бути в кодуванні слів типу `I'll` (токенізується в два токени `I 'll` чи в один токен `I'll`?), `don't`, `I've` і подібних.

Перевіримо, який варіант токенізації використовує GloVe:

In [42]:
"don't" in glove

False

In [43]:
"n't" in glove

True

In [44]:
"I'll" in glove

False

In [45]:
"'ll" in glove

True

Отже, маємо розбивати `don't` на два токени: `do` + `n't`

In [46]:
import spacy
from typing import List

spacy_nlp = spacy.blank("en")


def tokenize(text: str) -> List[str]:
  """
      Tokenize string with SpaCy.
  """

  tokens = spacy_nlp.tokenizer(text)
  return [str(token).lower() for token in tokens]

tokenize("I don't know")

['i', 'do', "n't", 'know']

## Векторизація одного документа

Тепер можемо порахувати вектор для кожного документу в корпусі. Цей вектор буде дорівнювати середньому від векторів окремих слів документа.

In [47]:
import numpy as np
from typing import Tuple

def bag_of_embeddings(doc: str, embeddings: KeyedVectors) -> np.ndarray:
    tokens = tokenize(doc)
    doc_vector = np.zeros(embeddings.vector_size)
    count = 0

    for token in tokens:
        if token in embeddings:
            doc_vector += embeddings[token]
            count += 1

    if count > 0:
        doc_vector /= count

    return doc_vector

doc_embedding = bag_of_embeddings("Hello world!", glove)
print(f"Embedding: {doc_embedding}")
print(f"Shape:     {doc_embedding.shape}")

Embedding: [ 0.41083999  0.49570001 -0.35982667 -0.40393333 -0.19768667  0.25433
 -0.5149      0.086394    0.04418867  0.36530902 -0.35623667  0.15675334
  0.0924      0.4017      0.00335333 -0.08881667 -0.37311666  0.57041667
  0.04311933  0.34267666  0.47882667  1.48930599  0.25857667 -0.19864706
  0.10432334  0.125766    0.10343501 -0.28578333 -0.31660533  0.013828
 -0.07466667  0.13855    -0.32214699 -0.28048033 -0.41403     0.06308667
 -0.23955599 -0.03680334 -0.141137   -0.0916     -0.16916067  0.57163533
 -0.31253667  0.07347333 -0.16953167  0.20232     0.60658666  0.06445
 -0.01957     0.30073999  0.21663668 -0.06867134  0.40304667  0.07233367
 -0.17247333  0.20601167 -0.39143999 -0.02245933 -0.17295867 -0.19998033
  0.234197   -0.29320667 -0.13344    -0.1644     -0.00662334  0.01097
 -0.00953534  0.50209667  0.42084     0.03766001  0.46454333 -0.44661133
  0.22226367  0.0327     -0.43125998  0.00774    -0.15634833  0.38859933
 -0.38001334 -0.13208101  0.22859666 -0.39621334  0

Розмірність вектора документа не залежить від кількості слів у ньому:

In [48]:
tests = [
    "Hello world",
    "You can try the best you can. The best you can is good enough.",
]

print("Розмір\tДокумент")

for s in tests:
    shape = bag_of_embeddings("Hello world!", glove).shape
    print(f"{shape}\t{s}")

Розмір	Документ
(200,)	Hello world
(200,)	You can try the best you can. The best you can is good enough.


In [49]:
lab.checkpoint("`Hello world` centroid", bag_of_embeddings("Hello world!", glove).mean())

Відповідь правильна ✅
Так, вірно! 💪


np.float64(0.0046237471218531275)

# Векторизація всього корпусу

Наступна операція може зайняти пару хвилин:

In [50]:
import datasets
imdb = datasets.load_dataset("stanfordnlp/imdb")

valid_data = imdb["test"].shuffle(seed=1).filter(lambda x, i: i < 2000, with_indices=True)  # take 2000 random rows for validation
train_data = imdb["train"].shuffle(seed=2)

In [51]:
from tqdm import tqdm


def vectorize_dataset(dataset: datasets.Dataset) -> Tuple[np.ndarray, np.ndarray]:
    """
        Векторізує весь датасет у представлення bag-of-embeddings.

        Повертає матрицю ознак X та вектор класів y.
    """
    
    X = []
    for doc in tqdm(dataset):
        doc_vector = bag_of_embeddings(doc["text"], glove)
        X.append(doc_vector)

    X = np.stack(X)
    y = np.array(dataset["label"])
    return (X, y)

In [52]:
X_train_boe, y_train_boe = vectorize_dataset(train_data)
X_valid_boe, y_valid_boe = vectorize_dataset(valid_data)
lab.checkpoint("Vectorized dataset shape", X_train_boe.shape)


  0%|          | 0/25000 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [00:03<00:00, 597.19it/s]


Відповідь правильна ✅
Так, вірно! 💪


(25000, 200)

## Logistic regression + Bag-of-Embeddings


In [53]:
# Штучно обмежимо кількість тренувальних прикладів цим значенням.
# Так ми емулюємо ситуацію, коли в нас мало тренувальних даних.
TRAIN_SIZE = 500

In [54]:
from sklearn.linear_model import LogisticRegression

# Тренуємо логістичну регресію
log_reg = LogisticRegression(solver="liblinear")

log_reg.fit(X_train_boe[:TRAIN_SIZE,], y_train_boe[:TRAIN_SIZE,])
log_reg_acc = log_reg.score(X_valid_boe, y_valid_boe)
lab.checkpoint(f"LogReg + BoE accuracy on {TRAIN_SIZE}", log_reg_acc)

0.7325

## Logistic regression + TF-IDF

Натренуємо для порівняння модель на TF-IDF bag-of-ngrams ознаках. Тренувальні дані точнісінько такі, як і у моделі bag-of-embeddings. Але як щодо якості?


In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_bow = vectorizer.fit_transform(train_data[:TRAIN_SIZE]["text"])

model_tfidf = LogisticRegression(solver='liblinear', C=0.2, penalty="l1")
model_tfidf.fit(X_train_bow, train_data["label"][:TRAIN_SIZE])

X_valid_bow = vectorizer.transform(valid_data["text"])
y_valid_bow = valid_data["label"]
tfidf_acc = model_tfidf.score(X_valid_bow, y_valid_bow)
lab.checkpoint(f"LogReg + TF-IDF accuracy on {TRAIN_SIZE}", tfidf_acc)

0.5035

# Завдання

Перетренуйте моделі на різних розмірах `TRAIN_SIZE`. Спробуйте кілька значень. Зверніть увагу, як різницю між моделями змінюється в залежності від `TRAIN_SIZE`.


In [56]:
sizes = [250, 500, 1000, 1500, 2000]
results = []

In [57]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for size in sizes:
    subset_train = train_data.select(range(size))
    X_train_boe, y_train_boe = vectorize_dataset(subset_train)
    X_valid_boe, y_valid_boe = vectorize_dataset(valid_data)

    model_boe = LogisticRegression(solver="liblinear")
    model_boe.fit(X_train_boe, y_train_boe)

    y_pred_boe = model_boe.predict(X_valid_boe)

    results.append({
        "train_size": size,
        "method": "LR+BoE",
        "accuracy": accuracy_score(y_valid_boe, y_pred_boe),
        "precision": precision_score(y_valid_boe, y_pred_boe, average="binary"),
        "recall": recall_score(y_valid_boe, y_pred_boe, average="binary"),
        "f1": f1_score(y_valid_boe, y_pred_boe, average="binary"),
    })

  0%|          | 0/250 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [00:03<00:00, 622.17it/s]


In [58]:
for size in sizes:
    subset_train = train_data.select(range(size))
    X_train_boe, y_train_boe = vectorize_dataset(subset_train)
    X_valid_boe, y_valid_boe = vectorize_dataset(valid_data)

    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X_train_bow = vectorizer.fit_transform(subset_train["text"])
    X_valid_bow = vectorizer.transform(valid_data["text"])

    model_tfidf = LogisticRegression(solver='liblinear', C=0.2, penalty="l1")
    model_tfidf.fit(X_train_bow, subset_train["label"])

    acc_tfidf = model_tfidf.score(X_valid_bow, valid_data["label"])
    results.append({
        "train_size": size,
        "method": "LR+TF-IDF",
        "accuracy": acc_tfidf
    })

100%|██████████| 2000/2000 [00:03<00:00, 633.67it/s]


In [59]:
import pandas as pd

df = pd.DataFrame(results)
print(df.pivot(index="train_size", columns="method", values="accuracy").sort_index())

method      LR+BoE  LR+TF-IDF
train_size                   
250         0.7130     0.5035
500         0.7325     0.5035
1000        0.7730     0.5035
1500        0.7880     0.5615
2000        0.7940     0.6370


# Embeddings matrix

Досі для доступу до векторів слів ми користувалися бібліотекою `gensim`, яка надавала нам інтерфейс словника (`dict`).

Під капотом, вектори слів зберігаються в одній матриці розмірності $|V| \times d$, де $|V|$ це розмір словника (скільки слів маємо), а $d$ — розмір вектора слова (в цій лабораторній було $d=200$)

In [60]:
glove.vectors.shape

(50000, 200)

В моделях глибинного навчання, як правило, справу мають саме з цією embeddings matrix.

Розглянемо два способи отримати вектор потрібного слова з цієї матриці:

## Vector-matrix multiplication

Перший спосіб, це представити слово з індексом $i$ у вигляді one-hot вектора $o_i$. Тоді ембедінг потрібного слова можна отримати в результаті добутку

$$e_i = \text{E}^\intercal o_i $$

In [61]:
import torch

embeddings_matrix = torch.tensor(glove.vectors)

def embed(token_index: int, embeddings_matrix: torch.tensor) -> torch.tensor:
    vocab_size, embed_dim = embeddings_matrix.shape
    one_hot = torch.zeros(vocab_size)
    one_hot[token_index] = 1
    return one_hot @ embeddings_matrix

assert torch.allclose(
    embed(42, embeddings_matrix),
    torch.tensor(glove.vectors[42]))


In [62]:
lab.checkpoint("The embedding of one-hot multiplication",
               embed(42, embeddings_matrix).sum())

'tensor(5.3482)'

## nn.Embedding

В PyTorch, як і в більшості deep learning frameworks, є спеціальна функція, яка повертає вектор з потрібним номером: `torch.nn.Embedding`. Вона імплементована більш ефективно, ніж спосіб з vector-matrix multiplication, тож в більшості випадків користуватися варто саме `nn.Embedding`.

In [63]:
import torch
from torch import nn

embeddings = nn.Embedding(num_embeddings=50_000, embedding_dim=200, _weight=embeddings_matrix)
indexes = torch.LongTensor([42])
embedded = embeddings(indexes)

assert np.isclose(embedded.sum().item(), glove.vectors[42].sum())

In [64]:
lab.checkpoint("nn.Embeddings", embedded.sum().item())

5.34816837310791

# Готово!

In [65]:
lab.answer("ALL DONE! 😊")

Відповідь правильна ✅
💪



'ALL DONE! 😊'